# BAN6800 — BA-08: Build Baseline Classifier

**Project:** AI-Enabled Predictive and Prescriptive Analytics for Marine Asset Maintenance and Operational Availability at Chairborne Global Services Limited (CGSL)

**Baseline model:** Logistic Regression  
**Purpose:** Establish a reproducible baseline benchmark before comparing subsequent classifiers.

### Methodological controls
- The official UCI APS test set is **not used for model selection or threshold tuning** in BA-08.
- A stratified validation split is created from the training data only.
- Scaling is fitted inside a pipeline on the training split only.
- Class imbalance is addressed using `class_weight="balanced"`.
- Classification threshold remains at the default **0.50** for the baseline. Threshold tuning is reserved for BA-10.
- Reported metrics are validation metrics and are not presented as production performance for CGSL.


In [ ]:
# BA-08 — Setup

import os
import json
import warnings
from io import BytesIO

import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

print("BA-08 libraries loaded successfully.")


## 1. Load the validated BA-07 feature artifacts

The final BA-07 feature matrix contains **334 features** after removal of the zero-variance feature `cd_000`.

The large NumPy arrays are stored in Git LFS. The code below retrieves the actual LFS objects from GitHub's media endpoint rather than reading the Git LFS pointer files.


In [ ]:
# URLs for the BA-07 final feature arrays stored through Git LFS

TRAIN_FEATURE_URL = "https://media.githubusercontent.com/media/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/X_train_final.npy"
TEST_FEATURE_URL = "https://media.githubusercontent.com/media/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/X_test_final.npy"

def load_npy_from_url(url, label):
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    content_type = response.headers.get("content-type", "")
    print(f"{label}: HTTP {response.status_code}; {len(response.content)/1024**2:.1f} MB received; content-type={content_type}")
    return np.load(BytesIO(response.content), allow_pickle=False)

X_train = load_npy_from_url(TRAIN_FEATURE_URL, "X_train_final")
X_test = load_npy_from_url(TEST_FEATURE_URL, "X_test_final")

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)


In [ ]:
# Load processed target labels from the GitHub repository

Y_TRAIN_URL = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/y_train.csv"
Y_TEST_URL = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/y_test.csv"

y_train = pd.read_csv(Y_TRAIN_URL).squeeze("columns")
y_test = pd.read_csv(Y_TEST_URL).squeeze("columns")

print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts())


In [ ]:
# Basic artifact integrity checks

assert X_train.shape == (60000, 334), f"Unexpected X_train shape: {X_train.shape}"
assert X_test.shape == (16000, 334), f"Unexpected X_test shape: {X_test.shape}"
assert len(y_train) == 60000, f"Unexpected y_train length: {len(y_train)}"
assert len(y_test) == 16000, f"Unexpected y_test length: {len(y_test)}"

assert np.isfinite(X_train).all(), "X_train contains non-finite values."
assert np.isfinite(X_test).all(), "X_test contains non-finite values."

print("Artifact integrity checks passed.")


## 2. Create a stratified training/validation split

Only the **training dataset** is split here. The official UCI test set is retained for later evaluation and is not used to choose the baseline model or its threshold.

The split is stratified to preserve the highly imbalanced positive/negative class proportions.


In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=VALIDATION_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

print("Model-fitting subset:", X_fit.shape)
print("Validation subset:", X_valid.shape)

print("\nClass distribution in model-fitting subset:")
print(y_fit.value_counts(normalize=True).rename("proportion").round(4))

print("\nClass distribution in validation subset:")
print(y_valid.value_counts(normalize=True).rename("proportion").round(4))


## 3. Train the baseline Logistic Regression model

Logistic Regression is used as the baseline because it provides a simple, interpretable reference model for binary classification.

`class_weight="balanced"` is applied because the training data contain only 1.67% positive cases. Standardization is fitted within the pipeline on the model-fitting subset only, preventing leakage from the validation subset.


In [ ]:
baseline_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                solver="lbfgs",
                max_iter=500,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

baseline_pipeline.fit(X_fit, y_fit)

print("Baseline Logistic Regression trained successfully.")


## 4. Evaluate the baseline on the validation set

The baseline threshold is **0.50**. Precision, recall, F1-score and ROC-AUC are reported, together with the confusion matrix and classification report.

These are validation results only.


In [ ]:
# Generate validation predictions

y_valid_pred = baseline_pipeline.predict(X_valid)
# Robustly select the probability column corresponding to the positive class.
model_classes = list(baseline_pipeline.named_steps["model"].classes_)
pos_index = model_classes.index("pos")
y_valid_prob = baseline_pipeline.predict_proba(X_valid)[:, pos_index]

precision = precision_score(y_valid, y_valid_pred, pos_label="pos", zero_division=0)
recall = recall_score(y_valid, y_valid_pred, pos_label="pos", zero_division=0)
f1 = f1_score(y_valid, y_valid_pred, pos_label="pos", zero_division=0)
roc_auc = roc_auc_score((y_valid == "pos").astype(int), y_valid_prob)

cm = confusion_matrix(
    y_valid,
    y_valid_pred,
    labels=["neg", "pos"]
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    cm,
    index=["Actual neg", "Actual pos"],
    columns=["Predicted neg", "Predicted pos"]
))

print("\nClassification report:")
print(classification_report(
    y_valid,
    y_valid_pred,
    labels=["neg", "pos"],
    zero_division=0
))


In [ ]:
# Save the confusion matrix as an image

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("BA-08 Baseline Logistic Regression — Validation Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], ["neg", "pos"])
plt.yticks([0, 1], ["neg", "pos"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.savefig("BA-08_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved: BA-08_confusion_matrix.png")


## 5. Create reproducible BA-08 artifacts

The trained baseline model is saved after the validation benchmark. The final saved model is fitted on the **full training dataset** using the already-established BA-07 feature representation; the validation metrics above remain the benchmark used for BA-08.

No official test-set performance is reported in this ticket.


In [ ]:
# Refit the baseline pipeline on all available training data
# after the validation benchmark has been recorded.

baseline_pipeline.fit(X_train, y_train)

joblib.dump(
    baseline_pipeline,
    "baseline_logistic_regression.joblib"
)

print("Saved: baseline_logistic_regression.joblib")


In [ ]:
# Save baseline metric results

results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Dataset": "Validation split from training data",
        "Validation_Size": VALIDATION_SIZE,
        "Random_State": RANDOM_STATE,
        "Class_Weight": "balanced",
        "Threshold": 0.50,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc,
        "True_Negatives": int(cm[0, 0]),
        "False_Positives": int(cm[0, 1]),
        "False_Negatives": int(cm[1, 0]),
        "True_Positives": int(cm[1, 1]),
        "Official_Test_Set_Used": False,
        "Threshold_Tuning_Used": False,
    }
])

results.to_csv("BA-08_baseline_results.csv", index=False)

print("Saved: BA-08_baseline_results.csv")
display(results)


In [ ]:
# Save the validation predictions for auditability

validation_predictions = pd.DataFrame({
    "Actual_Class": y_valid.to_numpy(),
    "Predicted_Class": y_valid_pred,
    "Predicted_Probability_Pos": y_valid_prob,
})

validation_predictions.to_csv(
    "BA-08_validation_predictions.csv",
    index=False
)

print("Saved: BA-08_validation_predictions.csv")


In [ ]:
# Save a concise model card / baseline report

model_card = f'''# BA-08 Baseline Logistic Regression Report

## Purpose
Establish a reproducible baseline classifier for APS failure prediction using the prepared UCI APS feature representation.

## Data
- Training observations: 60,000
- Test observations: 16,000 (reserved for later evaluation)
- Final BA-07 features: 334
- Training target: 59,000 `neg` and 1,000 `pos`
- Positive-class prevalence: {((y_train == "pos").mean() * 100):.2f}%

## Method
- Stratified 80/20 validation split from the training data.
- StandardScaler fitted within the training pipeline.
- Logistic Regression with `class_weight="balanced"`.
- Baseline classification threshold: 0.50.
- Random state: {RANDOM_STATE}.
- No threshold tuning in BA-08.
- Official UCI test set was not used for baseline selection or tuning.

## Validation Results
- Precision: {precision:.4f}
- Recall: {recall:.4f}
- F1-score: {f1:.4f}
- ROC-AUC: {roc_auc:.4f}

## Confusion Matrix
- True Negatives: {cm[0,0]}
- False Positives: {cm[0,1]}
- False Negatives: {cm[1,0]}
- True Positives: {cm[1,1]}

## Interpretation
The baseline provides an initial benchmark for later model comparison. Because the dataset is highly imbalanced, recall, precision, F1-score and ROC-AUC are emphasized rather than accuracy alone.

The results are proxy-dataset validation results and should not be interpreted as direct evidence of CGSL vessel or equipment failure performance. The UCI feature names are anonymized, so physical sensor interpretations are not assigned.

## Intended Next Use
BA-09 will compare alternative classifiers using a consistent validation framework. BA-10 will examine threshold and cost-sensitive evaluation.
'''

with open("BA-08_baseline_report.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("Saved: BA-08_baseline_report.md")


In [ ]:
# Save the run configuration for reproducibility

run_config = {
    "project_ticket": "BA-08",
    "model": "LogisticRegression",
    "class_weight": "balanced",
    "solver": "lbfgs",
    "max_iter": 500,
    "random_state": RANDOM_STATE,
    "validation_size": VALIDATION_SIZE,
    "baseline_threshold": 0.50,
    "test_set_used_for_selection": False,
    "threshold_tuning": False,
    "final_feature_count": int(X_train.shape[1]),
}

with open("BA-08_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("Saved: BA-08_run_config.json")


## 6. BA-08 conclusion

The baseline Logistic Regression model establishes the first quantitative benchmark for the project. The benchmark is based on a stratified validation split of the training data, with class imbalance explicitly addressed.

The official UCI test set remains untouched for this ticket, and threshold optimization is intentionally deferred to BA-10. This preserves a clean experimental sequence for subsequent model comparison and cost-sensitive evaluation.


In [ ]:
# Final artifact inventory

artifacts = [
    "baseline_logistic_regression.joblib",
    "BA-08_baseline_results.csv",
    "BA-08_confusion_matrix.png",
    "BA-08_validation_predictions.csv",
    "BA-08_baseline_report.md",
    "BA-08_run_config.json",
]

print("BA-08 artifact inventory:")
for name in artifacts:
    print(f"- {name}: {'READY' if os.path.exists(name) else 'MISSING'}")


In [ ]:
# Create a single downloadable BA-08 artifact bundle
# Run this cell after the full notebook finishes successfully.

from google.colab import files
import zipfile
import os

artifact_files = [
    "baseline_logistic_regression.joblib",
    "BA-08_baseline_results.csv",
    "BA-08_confusion_matrix.png",
    "BA-08_validation_predictions.csv",
    "BA-08_baseline_report.md",
    "BA-08_run_config.json",
]

missing = [f for f in artifact_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"These artifacts are missing: {missing}")

zip_name = "BA-08_artifacts.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in artifact_files:
        z.write(f)

print(f"Created {zip_name}")
files.download(zip_name)
